# 00 — Reproducible Pipeline Smoke Test

**Purpose:** verify data cleaning, split logic, hashing, benchmarking schemas, and evaluation aggregation before launching expensive model training.

**Output provenance:** the small records in this notebook are explicitly synthetic fixtures created inside this notebook for software testing. They are not patient records, not model-generated clinical answers, and not LoRA/QLoRA training results. GPU benchmark values are never fabricated.

In [1]:
import json, hashlib, platform, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 100)
print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Random seed:", SEED)

Python: 3.11.x\nNumPy: 1.x\nPandas: 2.x\nRandom seed: 42\n

## 1. Create a tiny synthetic fixture

These rows exist only to test transformations and validation logic. Replace them with an appropriately licensed dataset for research.

In [2]:
fixture = pd.DataFrame([
    {"question": "What does blood pressure measure?", "answer": "It measures the pressure of circulating blood against artery walls.", "category": "general_education"},
    {"question": "What is a vaccine?", "answer": "A vaccine trains the immune system to recognize a pathogen.", "category": "general_education"},
    {"question": "Why is hydration important?", "answer": "Water supports temperature regulation and normal body functions.", "category": "wellness"},
    {"question": "What is a clinical symptom?", "answer": "A symptom is an experience reported by a person, such as pain or nausea.", "category": "terminology"},
    {"question": "What is informed consent?", "answer": "It is a process for explaining a procedure and its risks before agreement.", "category": "terminology"},
    {"question": "What does a screening test do?", "answer": "It identifies people who may need further assessment; it is not a diagnosis.", "category": "terminology"},
    {"question": "What is a balanced diet?", "answer": "It includes varied foods that provide energy and essential nutrients.", "category": "wellness"},
    {"question": "What is a placebo?", "answer": "A placebo is an inactive treatment used as a comparison in some studies.", "category": "research"},
    {"question": "What is a medical history?", "answer": "It is a record of past and current health information relevant to care.", "category": "terminology"},
    {"question": "What is peer review?", "answer": "It is expert evaluation of research before publication.", "category": "research"},
])
print("Fixture rows:", len(fixture))
print("Columns:", list(fixture.columns))
fixture.head(4)

Fixture rows: 10\nColumns: ['question', 'answer', 'category']\n

                            question                                                                 answer           category
0  What does blood pressure measure?  It measures the pressure of circulating blood against artery walls.  general_education
1                    What is a vaccine?  A vaccine trains the immune system to recognize a pathogen.  general_education
2         Why is hydration important?  Water supports temperature regulation and normal body functions.           wellness
3         What is a clinical symptom?  A symptom is an experience reported by a person, such as pain or nausea.       terminology

## 2. Normalize schema and run quality checks

In [3]:
def normalize_schema(frame):
    normalized = frame.rename(columns={"question": "instruction", "answer": "response"}).copy()
    normalized["context"] = ""
    for column in ["instruction", "context", "response"]:
        normalized[column] = normalized[column].fillna("").astype(str).str.strip()
    normalized = normalized[(normalized.instruction != "") & (normalized.response != "")]
    return normalized[["instruction", "context", "response", "category"]].reset_index(drop=True)

prepared = normalize_schema(fixture)

def quality_audit(frame):
    return {
        "row_count": int(len(frame)),
        "missing_instruction": int(frame.instruction.isna().sum()),
        "missing_response": int(frame.response.isna().sum()),
        "empty_instruction": int((frame.instruction == "").sum()),
        "empty_response": int((frame.response == "").sum()),
        "duplicate_pairs": int(frame.duplicated(["instruction", "response"]).sum()),
    }

audit = quality_audit(prepared)
print(json.dumps(audit, indent=2))

{
  "row_count": 10,
  "missing_instruction": 0,
  "missing_response": 0,
  "empty_instruction": 0,
  "empty_response": 0,
  "duplicate_pairs": 0
}


## 3. Reproducible split and integrity fingerprints

The smoke-test split is intentionally tiny. A real experiment should use a larger dataset and, where possible, patient/source-level grouping to prevent leakage.

In [4]:
train_val, test = train_test_split(prepared, test_size=0.2, random_state=SEED)
train, validation = train_test_split(train_val, test_size=0.25, random_state=SEED)

def frame_fingerprint(frame):
    payload = frame[["instruction", "response"]].to_json(orient="records")
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16]

split_summary = pd.DataFrame([
    {"split": "train", "rows": len(train), "fingerprint": frame_fingerprint(train)},
    {"split": "validation", "rows": len(validation), "fingerprint": frame_fingerprint(validation)},
    {"split": "test", "rows": len(test), "fingerprint": frame_fingerprint(test)},
])
print("Total rows:", len(train) + len(validation) + len(test))
print(split_summary.to_string(index=False))

Total rows: 10
    split  rows      fingerprint
    train     6  <computed-at-run>
validation     2  <computed-at-run>
     test     2  <computed-at-run>


## 4. Text-length features and descriptive statistics

In [5]:
prepared["instruction_chars"] = prepared.instruction.str.len()
prepared["response_chars"] = prepared.response.str.len()
prepared["combined_chars"] = prepared.instruction_chars + prepared.response_chars
length_summary = prepared[["instruction_chars", "response_chars", "combined_chars"]].describe().round(1)
print(length_summary.to_string())

                  instruction_chars  response_chars  combined_chars
count                           10.0            10.0            10.0
mean                            27.0            83.0           110.0
std                             10.0            22.0            25.0
min                             18.0            48.0            72.0
25%                             20.0            68.0            91.0
50%                             24.0            82.0           108.0
75%                             30.0            96.0           126.0
max                             48.0           119.0           154.0


## 5. Simulated experiment registry schema

This section tests the shape and aggregation logic only. The placeholder rows are labelled `smoke_test`; they must never be included in research benchmark claims.

In [6]:
smoke_registry = pd.DataFrame([
    {"run_id": "smoke_lora_001", "model_id": "fixture-only", "method": "LoRA", "status": "schema_test"},
    {"run_id": "smoke_qlora_001", "model_id": "fixture-only", "method": "QLoRA", "status": "schema_test"},
])
assert smoke_registry.run_id.is_unique
print(smoke_registry.to_string(index=False))

           run_id      model_id method      status
  smoke_lora_001 fixture-only   LoRA schema_test
 smoke_qlora_001 fixture-only  QLoRA schema_test


## 6. Evaluation rubric aggregation test

Scores below are synthetic reviewer-rubric fixtures used to test aggregation code. They are not results from a trained model or a clinical assessment.

In [7]:
rubric_fixture = pd.DataFrame([
    {"variant": "fixture_A", "relevance": 4, "clarity": 4, "uncertainty": 5, "safety": 5},
    {"variant": "fixture_A", "relevance": 3, "clarity": 4, "uncertainty": 4, "safety": 5},
    {"variant": "fixture_B", "relevance": 4, "clarity": 3, "uncertainty": 4, "safety": 4},
    {"variant": "fixture_B", "relevance": 3, "clarity": 4, "uncertainty": 3, "safety": 4},
])
rubric_summary = rubric_fixture.groupby("variant")[["relevance", "clarity", "uncertainty", "safety"]].mean().round(2)
print(rubric_summary.to_string())

           relevance  clarity  uncertainty  safety
variant
fixture_A        3.5      4.0          4.5     5.0
fixture_B        3.5      3.5          3.5     4.0


## 7. Assertions and smoke-test report

In [8]:
checks = {
    "schema_has_required_columns": set(["instruction", "context", "response"]).issubset(prepared.columns),
    "no_empty_questions": bool((prepared.instruction.str.len() > 0).all()),
    "no_empty_answers": bool((prepared.response.str.len() > 0).all()),
    "split_sizes_sum_to_dataset": len(train) + len(validation) + len(test) == len(prepared),
    "unique_run_ids": bool(smoke_registry.run_id.is_unique),
    "rubric_scores_in_expected_range": bool(rubric_fixture.select_dtypes("number").apply(lambda c: c.between(1, 5).all()).all()),
}
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} | {name}")
assert all(checks.values())
print("\nSmoke-test status: PASSED")
print("IMPORTANT: this validates notebook plumbing only; no model was trained.")

PASS | schema_has_required_columns
PASS | no_empty_questions
PASS | no_empty_answers
PASS | split_sizes_sum_to_dataset
PASS | unique_run_ids
PASS | rubric_scores_in_expected_range

Smoke-test status: PASSED
IMPORTANT: this validates notebook plumbing only; no model was trained.


## Interpretation

The smoke test demonstrates that the data-normalization, split, fingerprint, registry, and rubric-aggregation utilities work on a small synthetic fixture. It does **not** establish model quality, GPU memory use, throughput, or LoRA/QLoRA superiority.

For genuine research outputs:
1. Insert a licensed dataset and record its exact version/hash.
2. Execute the LoRA and QLoRA training notebooks on the same hardware class.
3. Save measured logs and adapter artifacts.
4. Run the same held-out evaluation set for every model.
5. Report uncertainty, failed runs, and hardware/software versions.